# Part 2: Job Postings Analysis - Role Categorization and Requirements Extraction

This notebook uses **LangChain** with a **local Ollama model (`llama3.2`, a small language model)** to analyze job postings and produce, for each posting:

1. **Job Category Classification** — a broad domain (Technology/IT, Finance, Marketing, Healthcare, Education, etc.)
2. **Key Requirements Extraction** — required skills/technologies, education level, and experience

As in Part 1, we use Ollama rather than a hosted API to avoid rate limits and to make the full-dataset bonus feasible.

In [1]:
import pandas as pd
from typing import List, Literal
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

MODEL_NAME = "llama3.2"  # local SLM served via Ollama
llm = ChatOllama(model=MODEL_NAME, temperature=0)

print(f"Using Ollama model: {MODEL_NAME}")

Using Ollama model: llama3.2


## Step 1: Load the Dataset

Job postings dataset with `Job Title` and `Job Description` columns, scraped from online job boards.

In [2]:
raw_df = pd.read_csv("data/job_title_des.csv", index_col=0)
print(raw_df.shape)
raw_df.head(3)

(2277, 2)


,Job Title,Job Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."


In [3]:
df = raw_df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"}).copy()
df = df.head(25).reset_index(drop=True)
print(df.shape)
df.head()

(25, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,Full Stack Developer,job responsibility full stack engineer – react...


## Step 2: Job Category Classification

A prompt (with a few labeled examples) that maps a job title + description to a broad domain category, using `with_structured_output` for a clean single label. "Other" is used as a fallback when the role doesn't clearly fit.

In [4]:
CATEGORIES = [
    "Technology/IT", "Finance", "Marketing", "Healthcare", "Education",
    "Sales", "Human Resources", "Operations", "Design", "Customer Service", "Other",
]

class JobCategory(BaseModel):
    category: Literal[tuple(CATEGORIES)] = Field(description="The single best-fitting broad domain for this job")

classification_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given a job title and description, categorize the job into one of the following domains: "
     f"{', '.join(CATEGORIES)}. If unsure, use 'Other'. Respond with only the single best-fitting category.\n\n"
     "Examples:\n"
     "Job: Software Engineer. Description: Build and maintain backend services using Java and Spring Boot.\n"
     "Domain category: Technology/IT\n\n"
     "Job: Staff Accountant. Description: Prepare financial statements and reconcile accounts monthly.\n"
     "Domain category: Finance\n\n"
     "Job: Registered Nurse. Description: Provide patient care and administer medication in a hospital ward.\n"
     "Domain category: Healthcare"),
    ("human", "Job: {title}. Description: {description}\n\nDomain category:")
])

classification_chain = classification_prompt | llm.with_structured_output(JobCategory)

# --- Test on a sample datapoint ---
sample = df.iloc[0]
sample_result = classification_chain.invoke({"title": sample["Job_Title"], "description": sample["Job_Description"][:3000]})
print("Job Title:", sample["Job_Title"])
print("Predicted category:", sample_result.category)

Job Title: Flutter Developer
Predicted category: Technology/IT


## Step 3: Key Requirements Extraction

A single composite prompt extracts **Skills**, **Education**, and **Experience** in one call, using **JSON-mode prompting** (Ollama's `format="json"`) with a Pydantic schema validated via `JsonOutputParser`. In testing, LangChain's tool-calling `with_structured_output` frequently returned empty/blank fields on longer descriptions with this small model — JSON-mode prompting proved much more reliable. Missing information is explicitly represented as `"Not specified"` rather than left to guesswork.

In [5]:
class JobRequirements(BaseModel):
    skills: List[str] = Field(default_factory=list, description="Key skills, technologies, or tools mentioned; empty list if none")
    education: str = Field(description="Minimum/preferred education level, e.g. Bachelor's degree in CS; use 'Not specified' if not mentioned")
    experience: str = Field(description="Years of experience or experience level required, e.g. '3+ years'; use 'Not specified' if not mentioned")

requirements_parser = JsonOutputParser(pydantic_object=JobRequirements)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Extract the required skills/technologies, minimum education level, and required years of experience "
     "from the job description below. If a piece of information is not mentioned, use 'Not specified' "
     "(for education/experience) or an empty list (for skills). Do not invent information.\n{format_instructions}"),
    ("human", "Job Title: {title}\nJob Description:\n{description}")
]).partial(format_instructions=requirements_parser.get_format_instructions())

# JSON-mode (format="json") makes the Ollama model emit valid JSON directly, which this small
# model handles more reliably than LangChain's tool-calling with_structured_output.
requirements_llm = ChatOllama(model=MODEL_NAME, temperature=0, format="json")
extraction_chain = extraction_prompt | requirements_llm | requirements_parser

# --- Test on a sample datapoint ---
sample_reqs = extraction_chain.invoke({"title": sample["Job_Title"], "description": sample["Job_Description"][:4000]})
print("Job Title:", sample["Job_Title"])
print("Requirements:", sample_reqs)

Job Title: Flutter Developer
Requirements: {'skills': [], 'education': 'Not specified', 'experience': 'Not specified'}


## Step 4 & 5: Apply to All Postings and Update the DataFrame

In [6]:
def analyze_posting(title: str, description: str) -> dict:
    desc = str(description)[:4000]  # keep prompts within a reasonable context size for the SLM
    try:
        category = classification_chain.invoke({"title": title, "description": desc}).category
    except Exception:
        category = "Other"
    try:
        reqs = extraction_chain.invoke({"title": title, "description": desc})
        skills = reqs.get("skills", [])
        education = reqs.get("education", "Not specified")
        experience = reqs.get("experience", "Not specified")
    except Exception:
        skills, education, experience = [], "Not specified", "Not specified"
    return {
        "Predicted_Category": category,
        "Required_Skills": skills,
        "Education_Required": education,
        "Experience_Required": experience,
    }


def run_pipeline(data: pd.DataFrame) -> pd.DataFrame:
    results = []
    for _, row in tqdm(data.iterrows(), total=len(data), desc="Analyzing job postings"):
        results.append(analyze_posting(row["Job_Title"], row["Job_Description"]))
    results_df = pd.DataFrame(results)
    return pd.concat([data.reset_index(drop=True), results_df], axis=1)


final_df = run_pipeline(df)
final_df.head()

Analyzing job postings:   0%|          | 0/25 [00:00<?, ?it/s]

,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[],Not specified,Not specified
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, Django, API development (REST/RPC), L...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n...",Technology/IT,"[Machine Learning, Deep Learning, Python, Java]",Not specified,3+ years
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Ani...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, J Native, JavaScript, HTML, CSS, SASS/...",Not specified,5+ years


In [7]:
final_df.to_csv("outputs/part2_jobs_first25_results.csv", index=False)
final_df.to_json("outputs/part2_jobs_first25_results.json", orient="records", indent=2)
print("Saved to outputs/part2_jobs_first25_results.csv and .json")
final_df

Saved to outputs/part2_jobs_first25_results.csv and .json


,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[],Not specified,Not specified
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, Django, API development (REST/RPC), L...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n...",Technology/IT,"[Machine Learning, Deep Learning, Python, Java]",Not specified,3+ years
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Ani...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, J Native, JavaScript, HTML, CSS, SASS/...",Not specified,5+ years
5,Java Developer,Software Developer - Integration*\nImmediate O...,Technology/IT,"[Proven technical expertise in the design, dev...","Bachelor's Degree in Computer Science, Informa...",2 years
6,Full Stack Developer,senior full stack developer \- 1800026h cwt lo...,Technology/IT,"[NodeJS, Java, MongoDB, Elasticsearch, Redis, ...",B.Sc,2 years
7,JavaScript Developer,"Job Description:\n\nReactJS + NodeJs, Azure Fu...",Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL]",Not specified,"{'minimum': '3', 'maximum': '8'}"
8,DevOps Engineer,Main Responsibilities and Deliverables:\nManag...,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloud...",Not specified,Not specified
9,Software Engineer,"Overview\n\n\nBased in Silicon Valley, Tintri ...",Technology/IT,"[REST API, C/C++ for Linux/Unix, Python, Go, G...","BS or MS; computer engineering, computer scien...",Minimum 7 years of software development experi...


## Bonus: Full Dataset (all job postings)

Set `RUN_BONUS = True` below to process every posting in the dataset instead of just the first 25. This runs locally via Ollama with no external rate limits, but will take a while (roughly 1-2 hours depending on hardware, since each posting needs 2 LLM calls: classification and requirements extraction).

This cell is skipped by default so the notebook can be run top-to-bottom quickly.

In [8]:
RUN_BONUS = False

if RUN_BONUS:
    bonus_df = raw_df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"}).copy()

    bonus_results_df = run_pipeline(bonus_df)
    bonus_results_df.to_csv("outputs/part2_jobs_ALL_results.csv", index=False)
    bonus_results_df.to_json("outputs/part2_jobs_ALL_results.json", orient="records", indent=2)
    print("Saved to outputs/part2_jobs_ALL_results.csv and .json")
else:
    print("RUN_BONUS is False - skipping full-dataset run. Set RUN_BONUS = True to process every posting.")

RUN_BONUS is False - skipping full-dataset run. Set RUN_BONUS = True to process every posting.
